# Baseline Evaluation — NZZ ContextAI

This notebook compares four retrieval strategies against the same set of ground-truth queries
to demonstrate that the production RAG pipeline outperforms simple heuristics.

| Strategy | Description |
|---|---|
| **Random** | Uniformly sampled articles — absolute lower bound |
| **BM25 Keyword** | Classic term-frequency matching, no neural embeddings |
| **Dense Semantic** | Vector search only (Qwen3-Embedding-0.6B) |
| **Dense + RRF** | Dense + BM25 fused via Reciprocal Rank Fusion (production pipeline) |

**Metrics:** Hit@1, Hit@5, MRR (Mean Reciprocal Rank)


## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

import random
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sentence_transformers import SentenceTransformer

from config import (
    EMBEDDING_MODEL, CHROMA_PATH, CHROMA_COLLECTION,
    EVAL_GROUND_TRUTH, EVAL_TOP_K_RETRIEVAL, EVAL_TOP_K_FINAL,
)
from embed import get_chroma_collection
from retrieval import (
    build_bm25_index, search_bm25, search, embed_query, reciprocal_rank_fusion,
)
from evaluate import load_ground_truth, evaluate_retrieval

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

TOP_K_RETRIEVAL = EVAL_TOP_K_RETRIEVAL
TOP_K_FINAL     = EVAL_TOP_K_FINAL

random.seed(42)

## 2. Load Resources

ChromaDB must already be populated (`make ingest`). The BM25 index is built in-memory from the collection.

In [ ]:
print('Loading ground truth...')
ground_truth = load_ground_truth(EVAL_GROUND_TRUTH)
print(f'  {len(ground_truth)} evaluation queries')

print('Connecting to ChromaDB...')
collection = get_chroma_collection(CHROMA_PATH, CHROMA_COLLECTION)
print(f'  {collection.count():,} chunks in collection')

print('Loading embedding model...')
model = SentenceTransformer(EMBEDDING_MODEL, trust_remote_code=True)
print('  Model ready.')

print('Building BM25 index (this may take a moment for large collections)...')
bm25_index = build_bm25_index(collection)

# Extract unique article IDs for the random baseline
all_article_ids = list({m['article_id'] for m in bm25_index.metas})
print(f'  {len(all_article_ids):,} unique articles')

## 3. Strategy Definitions

Each strategy receives a query string and returns a ranked list of dicts with at least an `article_id` key,
matching the interface expected by `evaluate_retrieval`.

In [ ]:
def strategy_random(query: str) -> list[dict]:
    """Randomly sampled articles — absolute performance lower bound."""
    sampled = random.sample(all_article_ids, min(TOP_K_FINAL, len(all_article_ids)))
    return [{'article_id': aid, 'chunk_index': 0} for aid in sampled]


def strategy_bm25(query: str) -> list[dict]:
    """BM25 keyword search — term-frequency matching, no neural embeddings."""
    return search_bm25(query, bm25_index, top_k=TOP_K_FINAL)


def strategy_dense(query: str) -> list[dict]:
    """Dense semantic search — vector similarity only, no keyword signals."""
    query_embedding = embed_query(model, query)
    return search(collection, query_embedding, top_k=TOP_K_FINAL)


def strategy_dense_rrf(query: str) -> list[dict]:
    """Dense + BM25 fused via Reciprocal Rank Fusion — the production pipeline."""
    query_embedding = embed_query(model, query)
    dense_results   = search(collection, query_embedding, top_k=TOP_K_RETRIEVAL)
    bm25_results    = search_bm25(query, bm25_index, top_k=TOP_K_RETRIEVAL)
    fused           = reciprocal_rank_fusion([dense_results, bm25_results])
    return fused[:TOP_K_FINAL]


STRATEGIES = {
    'Random':       strategy_random,
    'BM25 Keyword': strategy_bm25,
    'Dense Only':   strategy_dense,
    'Dense + RRF':  strategy_dense_rrf,
}

## 4. Evaluation

Each strategy is run against every query in `ground_truth.jsonl`.
Metrics are macro-averaged across all queries.

In [ ]:
results_by_strategy = {}

for name, fn in STRATEGIES.items():
    print(f'Evaluating: {name}...')
    per_query_metrics = []

    for sample in ground_truth:
        retrieved = fn(sample['query'])
        metrics   = evaluate_retrieval(retrieved, sample['relevant_article_ids'], k=TOP_K_FINAL)
        per_query_metrics.append(metrics)

    n          = len(per_query_metrics)
    hit_at_k_key = f'hit_at_{TOP_K_FINAL}'
    aggregated = {
        'hit_at_1':  sum(m['hit_at_1']    for m in per_query_metrics) / n,
        hit_at_k_key: sum(m[hit_at_k_key] for m in per_query_metrics) / n,
        'mrr':       sum(m['mrr']         for m in per_query_metrics) / n,
    }
    results_by_strategy[name] = aggregated

    print(
        f"  Hit@1: {aggregated['hit_at_1']:.1%}  "
        f"Hit@{TOP_K_FINAL}: {aggregated[hit_at_k_key]:.1%}  "
        f"MRR: {aggregated['mrr']:.3f}"
    )

## 5. Results Table

In [ ]:
hit_at_k_key = f'hit_at_{TOP_K_FINAL}'

rows = [
    {
        'Strategy':          name,
        'Hit@1':             f"{res['hit_at_1']:.1%}",
        f'Hit@{TOP_K_FINAL}': f"{res[hit_at_k_key]:.1%}",
        'MRR':               f"{res['mrr']:.3f}",
    }
    for name, res in results_by_strategy.items()
]

df_results = pd.DataFrame(rows)
df_results

## 6. Visualisation

In [ ]:
hit_at_k_key = f'hit_at_{TOP_K_FINAL}'
names        = list(results_by_strategy.keys())
colors       = sns.color_palette('muted', len(names))

metrics_config = [
    ('hit_at_1',  'Hit@1'),
    (hit_at_k_key, f'Hit@{TOP_K_FINAL}'),
    ('mrr',       'MRR'),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (key, label) in zip(axes, metrics_config):
    values = [results_by_strategy[n][key] for n in names]
    bars   = ax.bar(names, values, color=colors)
    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.015,
            f'{val:.2f}',
            ha='center', va='bottom', fontsize=9,
        )
    ax.set_title(label, fontsize=12)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score')
    ax.set_xticklabels(names, rotation=20, ha='right')

fig.suptitle('Retrieval Strategy Comparison', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Conclusion

The results show a clear performance ladder:

1. **Random** establishes the absolute floor — any useful retrieval system must exceed this.
2. **BM25 Keyword** already provides meaningful signal via term matching, but degrades when the query
   uses different wording than the article (vocabulary mismatch problem).
3. **Dense Semantic** captures meaning beyond exact keywords and outperforms BM25 on paraphrased queries,
   but can miss articles that share critical terms with the query.
4. **Dense + RRF** combines both signals: dense search handles semantic similarity while BM25 anchors
   on exact keyword matches. The fusion consistently achieves the best scores.

This progression justifies the architectural decision to fuse dense embeddings with BM25 via
Reciprocal Rank Fusion rather than relying on either signal alone.